# 🎒 F0 — Generar Excel de prueba (50 alumnos)

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la Universitat Jaume I**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |
| **Versión** | AU_UJI Dinámico (V2) |
| **Fase** | 0 — Configuración |
| **Tipo** | Notebook auxiliar (uso opcional) |

---

## 🎯 ¿Qué hace este notebook?

Genera dos ficheros Excel **reducidos** (~50 alumnos) a partir de los Excel originales del proyecto, manteniendo **exactamente la misma estructura** (mismas hojas, mismas columnas, mismos tipos). El objetivo es que cualquier persona del tribunal o compañera/o de clase pueda **probar el pipeline completo** sin necesidad de los datos institucionales reales.

## 🔑 Cómo selecciona los 50 alumnos

Se hace un **muestreo estratificado por abandono/no-abandono** sobre la tabla `Expedientes`:
- ~25 alumnos que abandonaron (proxy: `exp_estudio_finalizado == 0`)
- ~25 alumnos que finalizaron

A partir de esos `per_id_ficticio` se filtran **todas las hojas** de los dos Excel.

## 📁 Qué genera

Dos ficheros nuevos en `data/00_raw/ejemplo/` (carpeta nueva, **no toca los originales**):
- `datos_proyecto_sin_preinscrip_DEMO.xlsx` (8 hojas, ~50 alumnos)
- `preinscripcion_si_DEMO.xlsx` (1 hoja `Hoja1`, mismos alumnos)

## 🧭 Flujo de uso para el tribunal

1. Quitar los 2 Excel originales de `data/00_raw/`.
2. Copiar los 2 Excel `_DEMO.xlsx` desde `data/00_raw/ejemplo/` a `data/00_raw/` y **renombrarlos** quitando el sufijo `_DEMO` (deben llamarse `datos_proyecto_sin_preinscrip.xlsx` y `preinscripcion_si.xlsx`).
3. Ejecutar `f0_validar_excel.ipynb` y luego `orquestador_maestro.ipynb`.

## ⚠️ Aviso

Este notebook **solo lo ejecuta María José en local** una única vez para generar los Excel reducidos que después se suben al repositorio. Una vez subidos a GitHub, el resto del mundo solo descarga el repo y los usa.

---

In [1]:
# ============================================================================
# CELDA 1 — CONFIGURACIÓN DE RUTAS (ROOT robusto)
# ============================================================================
# Detecta automáticamente la raíz del proyecto subiendo niveles hasta
# encontrar la carpeta src/. Idéntico patrón al resto de notebooks F0.
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

def _encontrar_root(start: Path) -> Path:
    """Sube por los padres hasta encontrar la carpeta src/."""
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError(f'No se encontró src/ subiendo desde {start}')

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

# --- Imports del proyecto ---
from src.config import (
    EXCEL_PRINCIPAL, EXCEL_PREINSCRIPCION,
    RUTA_RAW,
    MAPEO_HOJAS, HOJAS_EXCEL_PRINCIPAL,
    info_entorno,
)

info_entorno()

print(f'\n📂 ROOT del proyecto: {ROOT}')
print(f'📗 Excel principal:    {EXCEL_PRINCIPAL.name}')
print(f'📙 Excel preinscripción: {EXCEL_PREINSCRIPCION.name}')
print(f'📁 Carpeta data/00_raw/: {RUTA_RAW}')

✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     c:\FF\AU_UJI_v2
✓ 📁 RAW:           c:\FF\AU_UJI_v2\data\00_raw
✓ 📁 INTERIM:       c:\FF\AU_UJI_v2\data\01_interim
✓ 📁 PROCESSED:     c:\FF\AU_UJI_v2\data\02_processed
✓ 📁 FEATURES:      c:\FF\AU_UJI_v2\data\03_features
✓ 📁 AUTOML:        c:\FF\AU_UJI_v2\data\automl
✓ 📁 NOTEBOOKS:     c:\FF\AU_UJI_v2\notebooks
✓ 📄 Excel principal: c:\FF\AU_UJI_v2\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓ ===========================================================================

📂 ROOT del proyecto: c:\FF\AU_UJI_v2
📗 Excel principal:    datos_proyecto_sin_preinscrip.xlsx
📙 Excel preinscripción: preinscripcion_si.xlsx
📁 Carpeta data/00_raw/: c:\FF\AU_UJI_v2\data\00_raw


In [2]:
# ============================================================================
# CELDA 2 — VERIFICAR QUE EXISTEN LOS 2 EXCEL ORIGINALES
# ============================================================================
# Sin los Excel originales no se puede generar la versión reducida.
# Si alguno falta, el notebook se interrumpe con un mensaje claro.
# ============================================================================

print('=' * 60)
print('VERIFICACIÓN DE EXCEL ORIGINALES')
print('=' * 60)

if not EXCEL_PRINCIPAL.exists():
    raise FileNotFoundError(
        f'❌ No se encuentra: {EXCEL_PRINCIPAL}\n'
        f'   Coloca el Excel original en data/00_raw/ y vuelve a ejecutar.'
    )
if not EXCEL_PREINSCRIPCION.exists():
    raise FileNotFoundError(
        f'❌ No se encuentra: {EXCEL_PREINSCRIPCION}\n'
        f'   Coloca el Excel original en data/00_raw/ y vuelve a ejecutar.'
    )

tam_principal = EXCEL_PRINCIPAL.stat().st_size / (1024 * 1024)
tam_preins    = EXCEL_PREINSCRIPCION.stat().st_size / (1024 * 1024)

print(f'   ✅ {EXCEL_PRINCIPAL.name}: {tam_principal:,.1f} MB')
print(f'   ✅ {EXCEL_PREINSCRIPCION.name}: {tam_preins:,.1f} MB')

VERIFICACIÓN DE EXCEL ORIGINALES
   ✅ datos_proyecto_sin_preinscrip.xlsx: 27.8 MB
   ✅ preinscripcion_si.xlsx: 23.4 MB


In [3]:
# ============================================================================
# CELDA 3 — LEER LA HOJA "Expedientes" Y SELECCIONAR 50 ALUMNOS
# ============================================================================
# IMPORTANTE: los Excel originales vienen con columnas en mayúscula inicial
# (Per_id_ficticio, Exp_Tit_Id, Egresado, ...). La conversión a minúsculas
# y a 0/1 ocurre en la Fase 1 (limpieza). Aquí trabajamos con los nombres
# y valores CRUDOS del Excel.
#
# Detección automática de:
#   - Columna ID del alumno (Per_id_ficticio / per_id_ficticio).
#   - Columna de egresado (Egresado / egresado).
#   - Valores de egresado: pueden ser 'S'/'N', 'Si'/'No', 1/0, True/False.
# Muestreo estratificado 25/25, semilla fija (random_state=42).
# ============================================================================

import pandas as pd
import numpy as np

N_ALUMNOS_OBJETIVO = 50
SEMILLA = 42

print('=' * 60)
print('SELECCIÓN DE 50 ALUMNOS (muestreo estratificado)')
print('=' * 60)

df_exp = pd.read_excel(EXCEL_PRINCIPAL, sheet_name='Expedientes')
print(f'\n📖 Expedientes leídos: {len(df_exp):,} filas × {len(df_exp.columns)} columnas')
print(f'   Columnas: {list(df_exp.columns)}')

# --- Helper: buscar columna case-insensitive ---
def buscar_col(df, *posibles):
    """Devuelve el nombre real de la primera columna que coincida (sin importar mayúsculas)."""
    cols_lower = {c.lower(): c for c in df.columns}
    for p in posibles:
        if p.lower() in cols_lower:
            return cols_lower[p.lower()]
    return None

# --- Detectar columna ID del alumno ---
COL_ID = buscar_col(df_exp, 'per_id_ficticio')
if COL_ID is None:
    raise KeyError(
        f'No se encuentra columna per_id_ficticio en la hoja Expedientes. '
        f'Columnas disponibles: {list(df_exp.columns)}'
    )
print(f'\n🔑 Columna ID detectada: "{COL_ID}"')

# --- Detectar columna de egresado / graduación ---
COL_EGRESADO = buscar_col(df_exp, 'Egresado', 'egresado',
                          'exp_estudio_finalizado', 'estudio_finalizado',
                          'finalizado')

# --- Quedarnos con un alumno por per_id_ficticio (su última fila) ---
df_exp_unico = df_exp.drop_duplicates(subset=[COL_ID], keep='last')
print(f'\n   Alumnos únicos: {len(df_exp_unico):,}')

if COL_EGRESADO is None:
    print('\n⚠️  No se encuentra columna de egresado/finalización → muestreo aleatorio simple')
    todos_ids = df_exp_unico[COL_ID].unique()
    rng = np.random.default_rng(SEMILLA)
    ids_seleccionados = rng.choice(todos_ids, size=min(N_ALUMNOS_OBJETIVO, len(todos_ids)), replace=False)
    ids_abandono = ids_seleccionados[:len(ids_seleccionados)//2]
    ids_finalizado = ids_seleccionados[len(ids_seleccionados)//2:]
else:
    print(f'   📊 Columna de egresado detectada: "{COL_EGRESADO}"')
    print(f'   Valores distintos en la columna: {sorted(df_exp_unico[COL_EGRESADO].dropna().unique().tolist())}')
    print(f'   Distribución por alumno único:')
    print(df_exp_unico[COL_EGRESADO].value_counts(dropna=False).to_string())

    # ----- Detección automática de los valores SI/NO -----
    # Excel UJI usa 'S'/'N'; otros formatos: 'Si'/'No', 1/0, True/False, 'Y'/'N'.
    valores_si = {'S', 's', 'SI', 'Si', 'si', 'SÍ', 'Sí', 'sí',
                  'Y', 'y', 'YES', 'Yes', 'yes',
                  1, '1', True}
    valores_no = {'N', 'n', 'NO', 'No', 'no',
                  0, '0', False}

    serie_egresado = df_exp_unico[COL_EGRESADO]
    mask_finalizado = serie_egresado.isin(valores_si)
    mask_abandono   = serie_egresado.isin(valores_no)

    df_finalizado = df_exp_unico[mask_finalizado]
    df_abandono   = df_exp_unico[mask_abandono]

    print(f'\n   Tras normalización:')
    print(f'      No egresado (abandono):  {len(df_abandono):,} alumnos')
    print(f'      Sí egresado (finalizado): {len(df_finalizado):,} alumnos')

    if len(df_abandono) == 0 and len(df_finalizado) == 0:
        raise ValueError(
            f'No se han podido clasificar los valores de "{COL_EGRESADO}". '
            f'Valores encontrados: {serie_egresado.unique().tolist()}'
        )

    n_aban = min(N_ALUMNOS_OBJETIVO // 2, len(df_abandono))
    n_fin  = min(N_ALUMNOS_OBJETIVO - n_aban, len(df_finalizado))

    ids_abandono   = df_abandono.sample(n=n_aban, random_state=SEMILLA)[COL_ID].unique()
    ids_finalizado = df_finalizado.sample(n=n_fin, random_state=SEMILLA)[COL_ID].unique()
    ids_seleccionados = np.unique(np.concatenate([ids_abandono, ids_finalizado]))

print(f'\n✅ Alumnos seleccionados: {len(ids_seleccionados)}')
print(f'   - Con abandono:    {len(ids_abandono)}')
print(f'   - Con finalización: {len(ids_finalizado)}')

SELECCIÓN DE 50 ALUMNOS (muestreo estratificado)

📖 Expedientes leídos: 109,568 filas × 15 columnas
   Columnas: ['Per_id_ficticio', 'Exp_Tit_Id', 'Curso_Aca_Ini', 'Curso_Aca', 'Curso_Aca_Fin', 'Nota', 'Nombre', 'Seguro', 'Nota_selectividad', 'Nota_Acceso', 'Cred_Matriculados', 'Cred_Superados', 'Egresado', 'Nuevo', 'Media_Curso']

🔑 Columna ID detectada: "Per_id_ficticio"

   Alumnos únicos: 30,872
   📊 Columna de egresado detectada: "Egresado"
   Valores distintos en la columna: ['N', 'S']
   Distribución por alumno único:
Egresado
N    27419
S     3453

   Tras normalización:
      No egresado (abandono):  27,419 alumnos
      Sí egresado (finalizado): 3,453 alumnos

✅ Alumnos seleccionados: 50
   - Con abandono:    25
   - Con finalización: 25


In [4]:
# ============================================================================
# CELDA 4 — OBTENER exp_tit_id ASOCIADOS (para filtrar la hoja Titulaciones)
# ============================================================================
# La hoja "Titulaciones" no contiene per_id_ficticio (su clave es exp_tit_id).
# Para filtrarla coherentemente, extraemos los exp_tit_id de los expedientes
# de los 50 alumnos seleccionados (case-insensitive).
# ============================================================================

df_exp_filt = df_exp[df_exp[COL_ID].isin(ids_seleccionados)]

COL_TIT = buscar_col(df_exp, 'exp_tit_id', 'Exp_Tit_Id')
if COL_TIT and len(df_exp_filt) > 0:
    exp_tit_ids = df_exp_filt[COL_TIT].unique()
else:
    exp_tit_ids = []
    print('⚠️  No se encuentra columna exp_tit_id; la hoja Titulaciones se copiará íntegra.')

print(f'📋 Expedientes filtrados:    {len(df_exp_filt):,}')
print(f'📋 Titulaciones implicadas: {len(exp_tit_ids)}')

📋 Expedientes filtrados:    189
📋 Titulaciones implicadas: 29


In [5]:
# ============================================================================
# CELDA 5 — CREAR CARPETA DE SALIDA Y FILTRAR EXCEL PRINCIPAL
# ============================================================================
# Salida: data/00_raw/ejemplo/datos_proyecto_sin_preinscrip_DEMO.xlsx
# Cada hoja se filtra con detección dinámica (case-insensitive):
#   - Titulaciones → exp_tit_id
#   - Resto         → per_id_ficticio
# Si la hoja no tiene la columna esperada, se copia íntegra y se avisa.
# ============================================================================

RUTA_EJEMPLO = RUTA_RAW / 'ejemplo'
RUTA_EJEMPLO.mkdir(parents=True, exist_ok=True)

salida_principal = RUTA_EJEMPLO / 'datos_proyecto_sin_preinscrip_DEMO.xlsx'

print('=' * 60)
print('FILTRANDO EXCEL PRINCIPAL (8 hojas)')
print('=' * 60)

with pd.ExcelWriter(salida_principal, engine='openpyxl') as writer:
    for hoja in HOJAS_EXCEL_PRINCIPAL:
        df = pd.read_excel(EXCEL_PRINCIPAL, sheet_name=hoja)
        n_orig = len(df)

        if hoja == 'Titulaciones':
            col_t = buscar_col(df, 'exp_tit_id')
            if col_t and len(exp_tit_ids) > 0:
                df_filt = df[df[col_t].isin(exp_tit_ids)]
            else:
                df_filt = df
        else:
            col_pid = buscar_col(df, 'per_id_ficticio')
            if col_pid:
                df_filt = df[df[col_pid].isin(ids_seleccionados)]
            else:
                print(f'   ⚠️  {hoja}: sin columna per_id_ficticio, se copia íntegra')
                df_filt = df

        df_filt.to_excel(writer, sheet_name=hoja, index=False)
        print(f'   ✅ {hoja:30s}: {n_orig:>8,} → {len(df_filt):>5,} filas')

tam_demo = salida_principal.stat().st_size / 1024
print(f'\n💾 Guardado: {salida_principal.name} ({tam_demo:,.1f} KB)')

FILTRANDO EXCEL PRINCIPAL (8 hojas)
   ✅ Titulaciones                  :       45 →    29 filas
   ✅ Recibos                       :  114,454 →   192 filas
   ✅ Domicilios                    :  210,911 →   372 filas
   ✅ Expedientes                   :  109,568 →   189 filas
   ✅ Nac-Sexo_Nacionalidad         :   30,873 →    50 filas
   ✅ Circunstancias                :   70,524 →   101 filas
   ✅ Trabajo                       :  195,524 →   282 filas
   ✅ Notas                         :  107,908 →   194 filas

💾 Guardado: datos_proyecto_sin_preinscrip_DEMO.xlsx (50.7 KB)


In [6]:
# ============================================================================
# CELDA 6 — FILTRAR EXCEL DE PREINSCRIPCIÓN
# ============================================================================
# La hoja se llama 'Hoja1' (según TABLAS_INFO['preinscripcion']).
# Clave de filtrado: per_id_ficticio (case-insensitive).
# Algunos alumnos no aparecerán en preinscripción (no proceden de UJI o
# entraron por otra vía). Eso es esperable y NO es un error.
# ============================================================================

salida_preins = RUTA_EJEMPLO / 'preinscripcion_si_DEMO.xlsx'

print('=' * 60)
print('FILTRANDO EXCEL DE PREINSCRIPCIÓN')
print('=' * 60)

df_preins = pd.read_excel(EXCEL_PREINSCRIPCION, sheet_name=0)
nombre_hoja_preins = pd.ExcelFile(EXCEL_PREINSCRIPCION).sheet_names[0]
print(f'\n📖 Preinscripción leída: {len(df_preins):,} filas (hoja: "{nombre_hoja_preins}")')
print(f'   Columnas: {list(df_preins.columns)}')

col_pid_preins = buscar_col(df_preins, 'per_id_ficticio')
if col_pid_preins:
    df_preins_filt = df_preins[df_preins[col_pid_preins].isin(ids_seleccionados)]
else:
    print('   ⚠️  No se encuentra per_id_ficticio en preinscripción, se copia íntegra')
    df_preins_filt = df_preins

with pd.ExcelWriter(salida_preins, engine='openpyxl') as writer:
    df_preins_filt.to_excel(writer, sheet_name=nombre_hoja_preins, index=False)

tam_preins_demo = salida_preins.stat().st_size / 1024
print(f'   ✅ Filas filtradas: {len(df_preins):,} → {len(df_preins_filt):,}')
print(f'\n💾 Guardado: {salida_preins.name} ({tam_preins_demo:,.1f} KB)')

FILTRANDO EXCEL DE PREINSCRIPCIÓN

📖 Preinscripción leída: 210,986 filas (hoja: "Hoja1")
   Columnas: ['ANO', 'UNIVERSIDAD', 'MUNICIPIO', 'CP', 'ORDEN_TITULACION', 'CUPO', 'NOM_CUPO', 'ESTADO', 'NOTA_TXT', 'SOLICITUD_ID', 'COD_ESTUDIOS', 'VIA_ESTUDIOS', 'CONVOCATORIA', 'NOTA_1', 'NOTA_2', 'ANO_1', 'CON_1', 'TITULACION_CENTRO', 'ESTADO_TITULACION', 'UNI_ID', 'NOMBRE_UNIVERSIDAD', 'ESTUDIO', 'Per_id_ficticio', 'MATRICULADO']
   ✅ Filas filtradas: 210,986 → 285

💾 Guardado: preinscripcion_si_DEMO.xlsx (33.3 KB)


In [7]:
# ============================================================================
# CELDA 7 — RESUMEN Y SIGUIENTES PASOS
# ============================================================================

print('=' * 60)
print('GENERACIÓN COMPLETADA')
print('=' * 60)
print()
print('Ficheros generados en data/00_raw/ejemplo/:')
print(f'  ✓ {salida_principal.name} ({tam_demo:,.1f} KB)')
print(f'  ✓ {salida_preins.name} ({tam_preins_demo:,.1f} KB)')
print()
print('📌 Siguientes pasos para subirlos al repositorio:')
print('   1. Verificar que los DEMO se pueden leer con f0_validar_excel.ipynb')
print('      (renombrarlos temporalmente quitando _DEMO).')
print('   2. Hacer commit de la carpeta data/00_raw/ejemplo/ a GitHub.')
print('   3. Documentar la URL pública en docs/html/manual_inicio.html.')
print()
print('🔄 Para que el tribunal use los DEMO en lugar de los originales:')
print('   - Copiar los 2 ficheros DEMO a data/00_raw/')
print('   - Renombrar quitando el sufijo _DEMO')
print('   - Ejecutar f0_validar_excel.ipynb y orquestador_maestro.ipynb')

GENERACIÓN COMPLETADA

Ficheros generados en data/00_raw/ejemplo/:
  ✓ datos_proyecto_sin_preinscrip_DEMO.xlsx (50.7 KB)
  ✓ preinscripcion_si_DEMO.xlsx (33.3 KB)

📌 Siguientes pasos para subirlos al repositorio:
   1. Verificar que los DEMO se pueden leer con f0_validar_excel.ipynb
      (renombrarlos temporalmente quitando _DEMO).
   2. Hacer commit de la carpeta data/00_raw/ejemplo/ a GitHub.
   3. Documentar la URL pública en docs/html/manual_inicio.html.

🔄 Para que el tribunal use los DEMO en lugar de los originales:
   - Copiar los 2 ficheros DEMO a data/00_raw/
   - Renombrar quitando el sufijo _DEMO
   - Ejecutar f0_validar_excel.ipynb y orquestador_maestro.ipynb
